# 01 — Ingestion & Bronze Layer

## Purpose

Download OpenF1 data for configured seasons, save **raw Bronze JSONL** with minimal transformation, and generate **ingestion evidence** (manifest, row counts, schema reports).

## Connection to grading rubric

| Pillar | This notebook |
|--------|----------------|
| Pipeline Architecture | Medallion Bronze landing, manifests, Colab-first paths |
| Data Quality & Cleaning | Bronze row counts, schema report, schema drift flags |

## Bronze endpoints (expanded)

| Endpoint | Role |
|----------|------|
| `session_result` | **Required** for Gold target `points_finish` (final classification position) |
| `starting_grid` | **Optional** — supports grid-position heuristic baseline (grid ≤ 10); may be empty or 404 for some sessions |
| Other session endpoints | Laps, pit, weather, position, race_control, drivers |
| `meetings`, `sessions` | Global calendar / session discovery |

Run **SMOKE_TEST** first, then set `SMOKE_TEST = False` for the full 2023–2025 Colab evidence run.

> **Official evidence:** The local developer smoke test is **not** the official project evidence. The MBA report must use artifacts generated from **Google Colab Pro Plus** execution of this notebook.


## Setup

Run `00_colab_setup.ipynb` first, or ensure the repo is cloned and `src/` is on `PYTHONPATH`.

### Google Drive persistence (recommended)

- Set `USE_GOOGLE_DRIVE = True` to write Bronze JSONL and evidence reports to Drive.
- This prevents data loss if Colab disconnects during long ingestion.
- **Full ingestion should use Drive persistence** for official MBA evidence.
- Code remains in `/content/openf1-big-data-pipeline`; outputs go to `/content/drive/MyDrive/openf1_big_data_pipeline` when Drive is enabled.
- Set `OPENF1_DATA_ROOT` **before** importing `openf1_pipeline.config`.


In [9]:
from pathlib import Path
import os
import sys
import importlib.util

# -----------------------------
# Colab-safe project setup
# -----------------------------
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# Optional but recommended: persistent outputs on Google Drive
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT set to:", os.environ["OPENF1_DATA_ROOT"])

# Clone or update repo
if not PROJECT_ROOT.exists():
    print("Cloning repo...")
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print("Repo already exists. Pulling latest changes...")
    %cd {PROJECT_ROOT}
    !git pull

# Move into repo
os.chdir(PROJECT_ROOT)
print("Current working directory:", Path.cwd())

# Install requirements
!pip install -q -r requirements.txt

# NOTE: !pip install -e . was removed because the project does not have a setup.py or pyproject.toml
# This means it's not a standard installable package.

# Add src to Python path to make local modules discoverable
SRC_PATH = PROJECT_ROOT / "src"
assert SRC_PATH.exists(), f"src folder not found: {SRC_PATH}"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Verify public root marker
assert (PROJECT_ROOT / "README.md").exists(), "README.md not found. Repo root may be wrong."

print("sys.path before import:", sys.path)

# Diagnostics to inspect the src directory content
print(f"\nContents of {SRC_PATH}:")
!ls -F {SRC_PATH}
print(f"\nContents of {SRC_PATH}/openf1_pipeline:")
!ls -F {SRC_PATH}/openf1_pipeline

# Further diagnostics before import
print("\nPython version:", sys.version)
print("Current working directory before import:", os.getcwd())

spec = importlib.util.find_spec("openf1_pipeline")
if spec:
    print(f"Found spec for openf1_pipeline: {spec.origin}")
else:
    print("No spec found for openf1_pipeline.")

# Explicitly print content of __init__.py and config.py
openf1_pipeline_path = SRC_PATH / "openf1_pipeline"
init_file = openf1_pipeline_path / "__init__.py"
config_file = openf1_pipeline_path / "config.py"

print(f"\nContent of {init_file}:")
if init_file.exists():
    try:
        with open(init_file, "r") as f:
            print(f.read()[:200]) # Print first 200 chars
    except Exception as e:
        print(f"Error reading {init_file}: {e}")
else:
    print(f"{init_file} does not exist.")

print(f"\nContent of {config_file}:")
if config_file.exists():
    try:
        with open(config_file, "r") as f:
            print(f.read()[:200]) # Print first 200 chars
    except Exception as e:
        print(f"Error reading {config_file}: {e}")
else:
    print(f"{config_file} does not exist.")

# Test import
import openf1_pipeline
print("openf1_pipeline import successful")
print("src path:", SRC_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OPENF1_DATA_ROOT set to: /content/drive/MyDrive/openf1_big_data_pipeline
Repo already exists. Pulling latest changes...
/content
Already up to date.
Current working directory: /content/openf1-big-data-pipeline
sys.path before import: ['/content/openf1-big-data-pipeline/src', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']

Contents of /content/openf1-big-data-pipeline/src:
openf1_pipeline/

Contents of /content/openf1-big-data-pipeline/src/openf1_pipeline:
bronze/    features/  ingestion/   modeling/	 quality/  utils/
config.py  gold/      __init__.py  __pycache__/  silver/

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0

In [8]:
from pathlib import Path
import os
import sys
import importlib.util

PROJECT_ROOT = Path("/content/openf1-big-data-pipeline")
SRC_PATH = PROJECT_ROOT / "src"
PACKAGE_PATH = SRC_PATH / "openf1_pipeline"

os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT.exists(), PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH.exists(), SRC_PATH)
print("PACKAGE_PATH:", PACKAGE_PATH.exists(), PACKAGE_PATH)
print("__init__.py:", (PACKAGE_PATH / "__init__.py").exists())
print("config.py:", (PACKAGE_PATH / "config.py").exists())

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

importlib.invalidate_caches()

print("sys.path first entries:", sys.path[:5])
print("find_spec:", importlib.util.find_spec("openf1_pipeline"))

import openf1_pipeline
print("Import successful:", openf1_pipeline)

PROJECT_ROOT: True /content/openf1-big-data-pipeline
SRC_PATH: True /content/openf1-big-data-pipeline/src
PACKAGE_PATH: True /content/openf1-big-data-pipeline/src/openf1_pipeline
__init__.py: True
config.py: True
sys.path first entries: ['/content/openf1-big-data-pipeline/src', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12']
find_spec: ModuleSpec(name='openf1_pipeline', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7a62046ee150>, origin='/content/openf1-big-data-pipeline/src/openf1_pipeline/__init__.py', submodule_search_locations=['/content/openf1-big-data-pipeline/src/openf1_pipeline'])
Import successful: <module 'openf1_pipeline' from '/content/openf1-big-data-pipeline/src/openf1_pipeline/__init__.py'>


In [10]:
import pandas as pd
import sys
from pathlib import Path
import importlib

# Re-ensure SRC_PATH is in sys.path just in case of state loss between cells
PROJECT_ROOT = Path("/content/openf1-big-data-pipeline")
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# If openf1_pipeline is already in sys.modules, remove it to force a fresh import
# This helps in cases of inconsistent state or if __init__.py itself has issues with previous partial imports.
if 'openf1_pipeline' in sys.modules:
    del sys.modules['openf1_pipeline']

# Now attempt the import, which should re-evaluate the module from the correct sys.path
from openf1_pipeline.config import (
    SEASONS,
    ENDPOINTS,
    ensure_project_directories,
    get_project_root,
    get_output_root,
    get_bronze_dir,
    get_manifests_dir,
    get_data_quality_reports_dir,
    get_schemas_dir,
)

paths = ensure_project_directories()

print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("BRONZE_DIR:", get_bronze_dir())
print("MANIFESTS_DIR:", get_manifests_dir())
print("DATA_QUALITY_REPORTS_DIR:", get_data_quality_reports_dir())
print("SCHEMAS_DIR:", get_schemas_dir())

PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
MANIFESTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests
DATA_QUALITY_REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
SCHEMAS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/schemas


In [ ]:
# Step 1: SMOKE_TEST = True (few sessions). Step 2: SMOKE_TEST = False (full evidence).
SMOKE_TEST = True
MAX_SESSIONS = 2 if SMOKE_TEST else None
INGEST_SEASONS = [2024] if SMOKE_TEST else SEASONS

print(f"SMOKE_TEST={SMOKE_TEST}, seasons={INGEST_SEASONS}, max_sessions={MAX_SESSIONS}")


## Run Bronze ingestion


In [ ]:
manifest_df = run_bronze_ingestion(
    seasons=INGEST_SEASONS,
    endpoints=ENDPOINTS,
    bronze_dir=get_bronze_dir(),
    manifests_dir=get_manifests_dir(),
    max_sessions=MAX_SESSIONS,
)

manifest_summary = summarize_manifest(manifest_df)
manifest_summary


## Endpoint status summary


In [ ]:
print("=== Manifest status counts ===")
print(manifest_summary["status_counts"])

print("\n=== Endpoints with at least one success ===")
print(manifest_summary["success_endpoints"])

print("\n=== Endpoints with failures (pipeline continued) ===")
print(manifest_summary["failed_endpoints"])

print("\n=== Row counts by endpoint (manifest) ===")
for ep, n in sorted(manifest_summary["row_counts_by_endpoint"].items()):
    print(f"  {ep}: {n}")

print("\n=== Target-support endpoints ===")
print(f"session_result total rows: {manifest_summary['session_result_total_rows']}")
print(f"starting_grid total rows: {manifest_summary['starting_grid_total_rows']}")

if manifest_summary["session_result_total_rows"] == 0 and not SMOKE_TEST:
    print("WARNING: session_result has zero rows — check manifest failures before Silver/Gold.")

manifest_df.groupby(["endpoint", "status"]).size().unstack(fill_value=0)


## Generate Bronze evidence reports


In [ ]:
report_result = generate_bronze_reports(
    bronze_dir=get_bronze_dir(),
    data_quality_reports_dir=get_data_quality_reports_dir(),
    schemas_dir=get_schemas_dir(),
)

report_result


## Inspect outputs


In [ ]:
row_counts = pd.read_csv(report_result["paths"]["bronze_row_counts"])
schema_report = pd.read_csv(report_result["paths"]["bronze_schema_report"])
schema_drift = pd.read_csv(report_result["paths"]["bronze_schema_drift"])

print("Row counts by endpoint (Bronze files):")
display(row_counts.groupby("endpoint")["row_count"].sum().reset_index())

print("\nSchema report (head):")
display(schema_report.head(10))

drift_flags = schema_drift[schema_drift["possible_schema_drift_flag"] == True]
print(f"Schema drift flags: {len(drift_flags)}")
if len(drift_flags):
    display(drift_flags.head(15))


## Checklist update notes

After a **successful Colab run**:

1. Confirm `artifacts/manifests/ingestion_manifest.csv` exists.
2. Confirm `session_result` has non-zero rows for full ingestion (or document gaps).
3. Note `starting_grid` row counts (may be zero for some sessions — not a failure if API returned success).
4. Update `implementation_checklist.md` — mark Colab smoke/full run only after Colab execution.
5. Next: `02_silver_cleaning_quality.ipynb`.
